# Validation Against Maynard et al. publised dataset

Validates Hotspot-derived malignant gene module scores (stem/differentiation/
inflammatory/basal/neuroendocrine/proliferative programs) against the public
Maynard-Bivona 2020 EGFR-mutant lung adenocarcinoma dataset, comparing
residual-disease (RD) vs. progressive-disease (PD) timepoints, separately for
EGFR-mutant and EGFR-wildtype patients.

## Setup

In [ ]:
import os

os.environ["NUMBA_CACHE_DIR"] = "/path/to/tmp"
os.environ["MPLCONFIGDIR"] = "/path/to/tmp"

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import stats
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(figsize=(4, 4))

## Config

In [ ]:
input_path = "/path/to/public/Maynard_Bivona_2020.h5ad"
metadata_path = "Maynard_metadata.csv"
hotspot_modules_csv = "./results/Hotspot/malignant_var2000_module_dense_subset_cluster.csv"
figures_output_dir = "./figures"

min_cells_per_sample = 5
timepoint_order = ["RD_Cancer cells", "PD_Cancer cells"]  # residual disease vs progressive disease

# marker gene panel for a quick sanity-check dotplot
marker_genes = {
    "MHC-classI": {"HLA-A", "HLA-B", "HLA-C"},
    "MHC-classII": {"HLA-DRA", "HLA-DRB1", "HLA-DQA1"},
    "Persister": {"AXL"},
}

# maps each score name to the Hotspot module IDs (from hotspot_modules_csv) that define it
module_definitions = {
    "stem_genes": [6.0, 4.0],
    "dif_genes": [5.0],
    "inflam_genes": [14.0, 9.0],
    "base_genes": [12.0, 7.0, 13.0, 10.0, 11.0, 15.0],
    "neurod_genes": [2.0, 3.0, 8.0],
    "pro_genes": [1.0],
}
score_names = list(module_definitions.keys())

# EGFR subsets to run this validation on. "exclude_value" is the EGFR_mutation
# value to filter OUT, so each group keeps everyone else.
egfr_groups = [
    {"label": "EGFRmutant", "exclude_value": "not mutated"},
    {"label": "EGFRwildtype", "exclude_value": "mutated"},
]

os.makedirs(figures_output_dir, exist_ok=True)

## Load and prepare data

In [ ]:
def load_and_prepare(input_path, metadata_path, min_cells_per_sample):
    """Load the public dataset, restrict to LUAD cancer cells, drop low-count
    samples, merge in study metadata, and label treatment-timepoint x cell-type."""
    adata = sc.read(input_path)
    adata = adata[adata.obs["disease"] == "lung adenocarcinoma"]
    adata = adata[adata.obs["ann_fine"] == "Cancer cells"]

    cell_counts = adata.obs["sample"].value_counts()
    valid_samples = cell_counts[cell_counts > min_cells_per_sample].index
    adata = adata[adata.obs["sample"].isin(valid_samples)].copy()

    metadata = pd.read_csv(metadata_path)
    metadata["sample"] = metadata["Sample Name"].apply(lambda x: f"Maynard_Bivona_2020_{x}")
    metadata = metadata.drop(columns=["Sample Name"])

    obs_merged = adata.obs.merge(metadata, on="sample", how="left")
    obs_merged.index = adata.obs_names
    adata.obs = obs_merged

    if not {"Treatement Timepoint", "ann_fine"}.issubset(adata.obs.columns):
        raise ValueError("Both 'Treatement Timepoint' and 'ann_fine' must be present in adata.obs.")
    adata.obs["tp_ann_fine"] = adata.obs["Treatement Timepoint"].astype(str) + "_" + adata.obs["ann_fine"].astype(str)

    # drop treatment-naive samples; this validation only concerns RD vs PD
    adata = adata[adata.obs["Treatement Timepoint"] != "TN"].copy()
    return adata

## Score Hotspot-derived gene modules

In [ ]:
def score_hotspot_modules(adata, hotspot_modules_csv, module_definitions, n_bins=25):
    """Score each cell for the given Hotspot gene modules."""
    modules_df = pd.read_csv(hotspot_modules_csv)
    for score_name, module_ids in module_definitions.items():
        gene_list = modules_df.loc[modules_df["Module"].isin(module_ids), "Gene"].tolist()
        sc.tl.score_genes(adata, gene_list=gene_list, n_bins=n_bins, score_name=score_name, use_raw=False)
    return adata

## Plot module scores by treatment timepoint

In [ ]:
def plot_module_violins(adata, score_names, timepoint_order, group_label, output_dir):
    """For each score, test RD vs PD (rank-sum, BH-corrected) and plot a violin."""
    df = adata.obs.copy()
    df["tp_ann_fine"] = pd.Categorical(df["tp_ann_fine"], categories=timepoint_order, ordered=True)

    for score in score_names:
        print(f"\n--- {score} ({group_label}) ---")

        p_values, comparisons = [], []
        for i in range(len(timepoint_order)):
            for j in range(i + 1, len(timepoint_order)):
                data1 = df.loc[df["tp_ann_fine"] == timepoint_order[i], score]
                data2 = df.loc[df["tp_ann_fine"] == timepoint_order[j], score]
                stat, pval = stats.ranksums(data1, data2)
                comparisons.append(f"{timepoint_order[i]} vs {timepoint_order[j]}")
                p_values.append(pval)

        adjusted_pvals = multipletests(p_values, method="fdr_bh")[1]
        for comparison, adj_pval in zip(comparisons, adjusted_pvals):
            print(f"Adjusted p-value for {comparison}: {adj_pval:.4e}")

        plt.figure(figsize=(1.5, 2.5))
        ax = sns.violinplot(x="tp_ann_fine", y=score, data=df, palette="Reds", linewidth=1)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
        plt.xlabel("Treatment Timepoint")
        plt.ylabel(score)
        plt.title(f"{score}_{group_label}")
        plt.savefig(os.path.join(output_dir, f"{score}_violin_{group_label}.pdf"), bbox_inches="tight")
        plt.show()

## Run validation for each EGFR subset

In [ ]:
for group in egfr_groups:
    label, exclude_value = group["label"], group["exclude_value"]
    print(f"\n===== {label} =====")

    adata = load_and_prepare(input_path, metadata_path, min_cells_per_sample)
    adata = adata[adata.obs["EGFR_mutation"] != exclude_value].copy()
    print("Cells:", adata.n_obs, "| EGFR_mutation values:", adata.obs["EGFR_mutation"].unique().tolist())

    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["logcounts"] = adata.X.copy()

    sc.pl.dotplot(
        adata, marker_genes, groupby="Treatement Timepoint", use_raw=False,
        standard_scale="var", cmap="Blues", figsize=(30, 7), dendrogram=True,
        save=f"_marker_sanity_check_{label}.pdf",
    )

    adata = score_hotspot_modules(adata, hotspot_modules_csv, module_definitions)
    plot_module_violins(adata, score_names, timepoint_order, group_label=label, output_dir=figures_output_dir)